# 📘 with문과 컴프리헨션

**with문**은 리소스를 안전하게 관리하고, **컴프리헨션**은 반복을 간결하게 표현합니다.

**학습 목표:**
- with문으로 파일/리소스 안전하게 관리
- 커스텀 컨텍스트 매니저 작성
- 리스트/딕셔너리/집합/제너레이터 컴프리헨션

## 1. with문 — 리소스 안전 관리

`with`문은 블록이 끝나면 **자동으로 리소스를 해제**합니다.
파일, 네트워크 연결, 락 등에 필수적입니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  with문 없이 파일 다루기 (권장하지 않음)      │
# │  f = open("file.txt", "w")                │
# │  f.write("hello")                          │
# │  f.close()  ← 예외 발생 시 close 안 됨!     │
# │                                            │
# │  with문 사용 (권장!)                        │
# │  with open("file.txt", "w") as f:           │
# │      f.write("hello")                      │
# │  ← 블록 종료 시 자동 close()               │
# └─────────────────────────────────────────┘

import os, tempfile
# with문으로 파일 쓰기
demo_file = os.path.join(tempfile.gettempdir(), "demo_with.txt")

with open(demo_file, "w") as f:
    f.write("Hello, World!\n")
    f.write("Using with statement\n")


In [ ]:
# ← 여기서 자동으로 close()

# with문으로 파일 읽기
with open(demo_file, "r") as f:
    content = f.read()
    print(content)


In [ ]:
# 여러 파일 동시 열기
copy_file = os.path.join(tempfile.gettempdir(), "demo_copy.txt")
with open(demo_file, "r") as src, open(copy_file, "w") as dst:
    dst.write(src.read())

with open(copy_file, "r") as f:
    print(f"복사된 내용: {f.read()}")
# ┌─────────────────────────────────────────┐
# │  with문의 핵심: 컨텍스트 매니저             │
# │  __enter__: 블록 진입 시 실행 (리소스 획득)  │
# │  __exit__: 블록 종료 시 실행 (리소스 해제)   │
# │  예외가 발생해도 __exit__은 항상 호출됨     │
# └─────────────────────────────────────────┘


## 2. 커스텀 컨텍스트 매니저

`__enter__`와 `__exit__` 메서드를 정의하면 어떤 클래스든 `with`문에서 사용할 수 있습니다.
`contextlib.contextmanager` 데코레이터로도 만들 수 있습니다.

In [ ]:
import time
# 클래스 기반 컨텍스트 매니저
class Timer:
    """실행 시간을 측정하는 컨텍스트 매니저"""
    def __enter__(self):
        self.start = time.time()
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time.time()
        self.duration = self.end - self.start
        print(f"실행 시간: {self.duration:.4f}초")
        return False  # 예외를 억제하지 않음

with Timer():
    total = sum(range(1_000_000))
    print(f"합계: {total}")


In [ ]:
# ┌─────────────────────────────────────────┐
# │  contextlib.contextmanager 데코레이터      │
# │  제너레이터 함수로 간단하게 만들 수 있음     │
# │  yield 앞: __enter__                      │
# │  yield 뒤: __exit__                       │
# └─────────────────────────────────────────┘

from contextlib import contextmanager

@contextmanager
def temp_value(value):
    """값을 임시로 제공하는 컨텍스트 매니저"""
    print(f"진입: {value}")
    try:
        yield value
    finally:
        print(f"종료: {value}")

with temp_value(42) as v:
    print(f"사용 중: {v}")


In [ ]:
# contextlib.suppress: 지정한 예외 무시
from contextlib import suppress

with suppress(FileNotFoundError):
    os.remove("/tmp/nonexistent_file.txt")
print("파일이 없어도 에러 없이 진행됨")


In [ ]:
# contextlib.closing: close() 메서드가 있는 객체
from contextlib import closing
from io import StringIO

with closing(StringIO()) as buf:
    buf.write("임시 버퍼")
    content = buf.getvalue()
    print(f"버퍼 내용: {content}")
# ← 자동으로 buf.close() 호출


## 3. 컴프리헨션(Comprehension)

컴프리헨션은 반복문과 조건문을 **한 줄로** 표현하는 파이썬 특유의 간결한 문법입니다.

```python
[표현식 for 변수 in iterable if 조건]       # 리스트
{키: 값 for 변수 in iterable if 조건}       # 딕셔너리
{표현식 for 변수 in iterable if 조건}       # 집합
(표현식 for 변수 in iterable if 조건)       # 제너레이터
```

In [ ]:
# ┌───────────────────────────────────────┐
# │  리스트 컴프리헨션                        │
# │  [표현식 for 변수 in iterable if 조건]   │
# └───────────────────────────────────────┘

# 기본
squares = [x ** 2 for x in range(1, 6)]
print(f"제곱: {squares}")            # [1, 4, 9, 16, 25]
# 조건 필터링
evens = [x for x in range(20) if x % 2 == 0]
print(f"짝수: {evens}")
# if-else 변환
labels = ["짝수" if x % 2 == 0 else "홀수" for x in range(5)]
print(f"라벨: {labels}")


In [ ]:
# 중첩 리스트 평탄화
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flat = [num for row in matrix for num in row]
print(f"평탄화: {flat}")


In [ ]:
# 문자열 처리
words = ["Hello World", "python is great", "  SPACES  "]
cleaned = [s.strip().lower() for s in words]
print(f"정제: {cleaned}")
# ┌───────────────────────────────────────┐
# │  딕셔너리 컴프리헨션                    │
# │  {키: 값 for 변수 in iterable if 조건} │
# └───────────────────────────────────────┘

squares_dict = {x: x ** 2 for x in range(1, 6)}
print(f"\n제곱 딕셔너리: {squares_dict}")


In [ ]:
# 두 리스트로 딕셔너리 만들기
keys = ["name", "age", "city"]
values = ["Alice", 30, "Seoul"]
person = {k: v for k, v in zip(keys, values)}
print(f"zip으로 생성: {person}")


In [ ]:
# 조건 필터링
scores = {"Alice": 85, "Bob": 62, "Charlie": 91, "Diana": 58}
passed = {name: score for name, score in scores.items() if score >= 70}
print(f"합격자: {passed}")


In [ ]:
# 키-값 교환
original = {"a": 1, "b": 2, "c": 3}
inverted = {v: k for k, v in original.items()}
print(f"키-값 교환: {inverted}")


In [ ]:
# ┌───────────────────────────────────────┐
# │  집합 컴프리헨션                        │
# │  {표현식 for 변수 in iterable if 조건}  │
# └───────────────────────────────────────┘

nums = [1, 2, 2, 3, 3, 3, 4, 4, 4, 4]
unique = {x for x in nums}
print(f"\n중복 제거: {unique}")

even_squares = {x ** 2 for x in range(20) if x % 2 == 0}
print(f"짝수 제곱: {sorted(even_squares)}")


In [ ]:
# 문자열에서 고유 문자
text = "hello world"
chars = {ch for ch in text if ch != " "}
print(f"고유 문자: {sorted(chars)}")


In [ ]:
# ┌───────────────────────────────────────┐
# │  제너레이터 표현식                       │
# │  (표현식 for 변수 in iterable if 조건)  │
# │  리스트 대신 제너레이터 반환 (메모리 절약) │
# └───────────────────────────────────────┘

gen = (x ** 2 for x in range(1, 6))
print(f"\n제너레이터: {gen}")
print(f"리스트 변환: {list(gen)}")


In [ ]:
# 메모리 효율 비교
import sys
list_comp = [x ** 2 for x in range(1000)]
gen_expr = (x ** 2 for x in range(1000))
print(f"리스트 크기: {sys.getsizeof(list_comp)} bytes")
print(f"제너레이터 크기: {sys.getsizeof(gen_expr)} bytes")


In [ ]:
# 제너레이터는 한 번만 순회 가능
gen2 = (x for x in range(5))
print(f"첫 순회: {list(gen2)}")    # [0, 1, 2, 3, 4]
print(f"두 번째: {list(gen2)}")    # [] (이미 소진됨!)


## 🎯 연습 문제

1. `with`문을 사용해 파일에 "파이썬 연습"을 쓰고 다시 읽어오는 코드를 작성하세요.
2. 리스트 컴프리헨션으로 1~100 중 3과 5의 공배수만 추출하세요.
3. 딕셔너리 컴프리헨션으로 단어와 그 길이를 매핑하는 딕셔너리를 만드세요.
4. `Timer` 컨텍스트 매니저를 사용해 `sum(range(1000000))`의 실행 시간을 측정하세요.